# Utils - QB - ImputationProvenanceTracker

Ce notebook illustre et vérifie le comportement de la classe
`ImputationProvenanceTracker` et de l'énumération `ProvenanceType`
(`tsforecast/frequency/provenance.py`), qui suivent la provenance de
chaque valeur d'un jeu de données imputé : donnée d'origine, valeur
issue d'un modèle (entraîné sur des données vraies ou mixtes), valeur
agrégée depuis une fréquence plus fine, ou valeur désagrégée depuis une
fréquence plus basse.

Le module ne contient **aucune fonction de niveau module** : seules
l'énumération `ProvenanceType` et les méthodes **publiques** de
`ImputationProvenanceTracker` sont testées ici :
`initialize`, `mark_imputed`, `mark_aggregated`, `mark_disaggregated`,
`mark_model_imputed`, `get_provenance`, `get_mask`, `compute_statistics`,
`get_provenance_matrix`, `to_string_matrix`, `merge`, `__repr__`. Les
attributs internes `_is_panel` et `_panel_cols` ne sont illustrés qu'au
travers de leurs effets observables (exclusion des colonnes de panel du
suivi, cf. §2.3).

**Contrat général** :
- La classe ne fait **aucune hypothèse sur la fréquence** des données :
  elle se contente d'un `DataFrame` quelconque (indexé par le temps ou
  non) et construit une matrice parallèle de même forme, où chaque
  cellule contient un `ProvenanceType` (ou `None`/`NaN` tant qu'elle n'a
  pas été marquée).
- `initialize()` marque automatiquement `ORIGINAL` toute valeur
  non-nulle du jeu de données d'entrée ; les valeurs manquantes restent
  à `None`, en attente d'un marquage explicite via les méthodes `mark_*`.
- `get_provenance_matrix()` et `to_string_matrix()` renvoient des copies :
  aucune méthode de lecture ne permet de modifier l'état interne du
  tracker.
- Toutes les méthodes (hors `initialize`) lèvent une `ValueError` si le
  tracker n'a pas été initialisé, ou si la colonne demandée n'existe pas
  dans la matrice de provenance.

## 1 - Import et instanciation

In [ ]:
# Importation des modules
import warnings

import numpy as np
import pandas as pd

# Classe et énumération testées
from tsforecast.frequency.provenance import ImputationProvenanceTracker, ProvenanceType

# Configuration de l'affichage
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore')

# Instanciation
tracker = ImputationProvenanceTracker()
print(tracker)

## 2 - Jeux de données

### 2.1 - Reprise des jeux de données de `3 - QB - Panel a frequences mixtes heterogene.ipynb`

Les deux fonctions génératrices sont recopiées telles quelles (convention
déjà suivie dans `frequency_aligner.ipynb`, `target_frequency_validator.ipynb`,
etc. : aucune fonction partagée n'existe entre notebooks dans ce projet)
pour obtenir :
- `df_timeseries` : indicateurs macroéconomiques mensuels/trimestriels/annuels,
  avec une variable annuelle (`balance_commerciale_annuelle`) dont
  l'historique démarre avant le début de la grille mensuelle (index
  irrégulier), et un délai de publication (dernière observation `NaN`)
  sur chaque variable.
- `df_panel` : panel de 3 pays (France, Allemagne, Italie), MultiIndex
  `(country, date)`, chaque pays ayant sa propre période de couverture,
  sa propre date de démarrage pour la production industrielle et sa
  propre fréquence de publication pour les dépenses publiques.

In [ ]:
# Fonction de création de séries temporelles (recopiée depuis le notebook 3)
def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    annual_start_date: str = '2015-01-01',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies.

    Args:
        start_date: Start date for the monthly variables of the dataset.
        end_date: End date for the dataset.
        annual_start_date: Start date for the annual trade balance series, earlier
            than `start_date` so that the resulting temporal index is irregular.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with mixed-frequency macroeconomic indicators.
    """
    # Initialisation du seed
    np.random.seed(seed)

    # Création de l'index mensuel
    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)

    # Initialisation du DataFrame
    df = pd.DataFrame(index=dates)
    df.index.name = 'date'

    # ----- Variables mensuelles -----
    # Production industrielle (mensuelle, croissance avec bruit)
    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise

    # Inflation mensuelle (IPC, entre 0.5% et 4%)
    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)

    # Taux de chômage (mensuel, entre 5% et 12%)
    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),  # Choc économique
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3))
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)

    # ----- Variable trimestrielle : PIB -----
    # Le PIB n'est disponible qu'aux fins de trimestre
    pib_base = 2500
    pib_growth_quarterly = 0.5  # Croissance trimestrielle moyenne
    df['pib_trimestriel'] = np.nan

    quarter_start_months = [1, 4, 7, 10]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_start_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1

    # ----- Variable annuelle : Balance commerciale -----
    # Historique disponible dès `annual_start_date`, antérieur au début des
    # variables mensuelles : l'index temporel global en devient irrégulier
    # (quelques observations annuelles isolées avant le début de la grille mensuelle).
    annual_dates = pd.date_range(start=annual_start_date, end=end_date, freq='YS')
    df = df.reindex(df.index.union(annual_dates))
    df.index.name = 'date'

    df['balance_commerciale_annuelle'] = np.nan
    for date in annual_dates:
        year_factor = (date.year - 2018)
        base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
        df.loc[date, 'balance_commerciale_annuelle'] = base_balance

    # ----- Simulation des délais de publication -----
    # Délai de 1 mois pour l'inflation et le chômage
    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan

    # Délai de 2 mois pour le PIB
    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

    # Délai de 3 mois pour la balance commerciale annuelle
    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

    # ----- Simulation de données historiques limitées -----
    # La production industrielle n'est disponible qu'à partir de 2019
    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan

    return df


# Création du jeu de données de séries temporelles
df_timeseries = create_timeseries_dataset()

print(f"Période : {df_timeseries.index.min().strftime('%Y-%m')} à {df_timeseries.index.max().strftime('%Y-%m')}")
print(f"Nombre d'observations : {len(df_timeseries)}")
print(f"Colonnes : {list(df_timeseries.columns)}")
print(f"Valeurs manquantes par colonne :\n{df_timeseries.isna().sum()}")

In [ ]:
# Fonction de création d'un jeu de données de panel fictif (recopiée depuis le notebook 3)
def create_panel_dataset(seed: int = 42) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies.

    Each entity has its own coverage period (start/end dates) and its own
    publication frequency for the public spending indicator, to simulate a
    heterogeneous panel across entities. The annual trade balance series
    also starts earlier than the other variables for each entity, making
    each entity's temporal index irregular.

    Args:
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with MultiIndex (country, date) and mixed-frequency indicators.
    """
    # Initialisation du seed
    np.random.seed(seed)

    # Définition des pays et leurs caractéristiques
    # Chaque pays possède sa propre période de couverture (start_date / end_date)
    # ainsi que sa propre fréquence de publication pour les dépenses publiques
    countries = {
        'France': {
            'pib_base': 2800,
            'inflation_base': 1.5,
            'chomage_base': 8.0,
            'depenses_base': 55.0,
            'start_date': '2018-01-01',
            'end_date': '2024-07-01',
            'prod_ind_start': '2018-06-01',  # Historique complet
            'depenses_frequency': 'annuelle',
            'annual_start_date': '2015-01-01'  # Historique de la balance commerciale antérieur au début mensuel
        },
        'Allemagne': {
            'pib_base': 3500,
            'inflation_base': 1.2,
            'chomage_base': 5.5,
            'depenses_base': 45.0,
            'start_date': '2018-07-01',  # Début plus tardif que la France
            'end_date': '2024-04-01',  # Fin plus précoce que la France
            'prod_ind_start': '2019-01-01',  # Historique partiel
            'depenses_frequency': 'trimestrielle',
            'annual_start_date': '2016-01-01'  # Historique de la balance commerciale antérieur au début mensuel
        },
        'Italie': {
            'pib_base': 2200,
            'inflation_base': 1.8,
            'chomage_base': 10.5,
            'depenses_base': 50.0,
            'start_date': '2019-01-01',  # Début encore plus tardif
            'end_date': '2024-07-01',
            'prod_ind_start': '2019-06-01',  # Historique plus court
            'depenses_frequency': 'annuelle',
            'annual_start_date': '2016-01-01'  # Historique de la balance commerciale antérieur au début mensuel
        }
    }

    # Initialisation de la liste des jeux de données pour l'ensemble des pays
    all_data = []

    # Parcours des pays
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)

        # Création de l'index de dates propre à ce pays (début/fin distincts)
        dates = pd.date_range(start=params['start_date'], end=params['end_date'], freq='MS')
        n_periods = len(dates)

        # Création du DataFrame pour ce pays
        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country

        # Production industrielle (mensuelle)
        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise

        # Données non disponibles avant une certaine date
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan

        # Inflation (mensuelle)
        infl_trend = np.linspace(
            params['inflation_base'],
            params['inflation_base'] + np.random.uniform(0.5, 2.0),
            n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)

        # Taux de chômage (mensuel)
        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3))
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)

        # PIB trimestriel
        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [1, 4, 7, 10]
        quarter_idx = 0
        for date in dates:
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1

        # Dépenses publiques (% du PIB) : fréquence annuelle ou trimestrielle selon le pays
        df_country['depenses_publiques_pib'] = np.nan
        if params['depenses_frequency'] == 'annuelle':
            publication_months = [1]
        else:
            publication_months = [1, 4, 7, 10]

        depenses_idx = 0
        for date in dates:
            if date.month in publication_months:
                value = params['depenses_base'] + 0.1 * depenses_idx + np.random.normal(0, 1.0)
                df_country.loc[date, 'depenses_publiques_pib'] = value
                depenses_idx += 1

        # Balance commerciale annuelle
        # Historique disponible dès `annual_start_date`, antérieur au début des
        # variables mensuelles de ce pays : l'index temporel de l'entité en
        # devient irrégulier.
        annual_dates = pd.date_range(start=params['annual_start_date'], end=params['end_date'], freq='YS')
        df_country = df_country.reindex(df_country.index.union(annual_dates))
        df_country['country'] = country

        df_country['balance_commerciale_annuelle'] = np.nan
        for date in annual_dates:
            year_factor = (date.year - 2018)
            base = -20 + np.random.uniform(-10, 10) + year_factor * 2
            df_country.loc[date, 'balance_commerciale_annuelle'] = base

        # Simulation des délais de publication
        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan

        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

        bc_available = df_country[df_country['balance_commerciale_annuelle'].notna()].index
        if len(bc_available) > 0:
            df_country.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

        depenses_available = df_country[df_country['depenses_publiques_pib'].notna()].index
        if len(depenses_available) > 0:
            df_country.loc[depenses_available[-1], 'depenses_publiques_pib'] = np.nan

        all_data.append(df_country)

    # Concaténation et création du MultiIndex
    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()

    return df_panel


# Création du jeu de données de panel
df_panel = create_panel_dataset()

print(f"Entités (pays) : {df_panel.index.get_level_values('country').unique().tolist()}")
print(f"Colonnes : {list(df_panel.columns)}")
print(f"Shape : {df_panel.shape}")

### 2.2 - `ImputationProvenanceTracker.initialize()` attend un `DataFrame` à colonnes,
pas un MultiIndex

`initialize()` exclut les colonnes listées dans `panel_cols` du suivi (ce
sont des identifiants d'entité, pas des variables à imputer), mais ne sait
rien faire d'un `MultiIndex` : l'usage réel (cf. `high_frequency_imputer.py`,
paramètre `panel_cols` du `HighFrequencyImputer`) consiste à passer un
DataFrame où l'entité est une **colonne ordinaire**, obtenu ici via
`reset_index(level='country')`.

In [ ]:
# Panel "à plat" : l'entité redevient une colonne ordinaire, l'index reste temporel
df_panel_flat = df_panel.reset_index(level='country')
display(df_panel_flat.head(3))
print()
print("dtypes de l'index :", type(df_panel_flat.index))

## 3 - `ProvenanceType`

Énumération `str` (`class ProvenanceType(str, Enum)`) : chaque membre se
compare donc directement à sa valeur `str` grâce à l'héritage double.

In [ ]:
for member in ProvenanceType:
    print(f"{member!r:45} value={member.value!r:15} == '{member.value}' -> {member == member.value}")

## 4 - `initialize(data, panel_cols=None)`

Construit la matrice de provenance : même index et mêmes colonnes que
`data` (moins `panel_cols`), valeurs non-nulles marquées `ORIGINAL`,
valeurs manquantes laissées à `None` (`dtype=object`, donc pas de
`np.nan` flottant implicite).

### 4.1 - Séries temporelles

In [ ]:
tracker_ts = ImputationProvenanceTracker()
tracker_ts.initialize(df_timeseries)

print("Colonnes suivies :", list(tracker_ts.provenance_matrix_.columns))
print("Shape matrice de provenance :", tracker_ts.provenance_matrix_.shape, "== shape des données :", df_timeseries.shape)
display(tracker_ts.provenance_matrix_.tail(8))

### 4.2 - Panel avec `panel_cols` : la colonne d'entité est exclue du suivi

In [ ]:
tracker_panel = ImputationProvenanceTracker()
tracker_panel.initialize(df_panel_flat, panel_cols=['country'])

print("Colonnes suivies :", list(tracker_panel.provenance_matrix_.columns))
print("'country' absente de la matrice de provenance :", 'country' not in tracker_panel.provenance_matrix_.columns)
display(tracker_panel.provenance_matrix_.tail(8))

### 4.3 - Cas limites

- `data` doit être un `DataFrame` (pas une `Series`, pas un array) ->
  `ValueError`.
- `data` ne peut pas être vide -> `ValueError`.
- Un `panel_cols` non trouvé dans `data` n'est **pas validé explicitement** :
  il est silencieusement ignoré par la compréhension de liste
  `[col for col in data.columns if col not in panel_cols]`, qui ne filtre
  que les colonnes réellement présentes.

In [ ]:
# data n'est pas un DataFrame
try:
    ImputationProvenanceTracker().initialize(df_timeseries['inflation_ipc'])
except ValueError as e:
    print("ValueError (Series) :", e)

# data est vide
try:
    ImputationProvenanceTracker().initialize(pd.DataFrame())
except ValueError as e:
    print("ValueError (DataFrame vide) :", e)

# panel_cols avec une colonne absente : aucune erreur, simplement ignorée
tracker_extra = ImputationProvenanceTracker()
tracker_extra.initialize(df_panel_flat, panel_cols=['country', 'colonne_absente'])
print("\nAucune erreur, colonnes suivies :", list(tracker_extra.provenance_matrix_.columns))

## 5 - `mark_imputed`, `mark_aggregated`, `mark_disaggregated`, `mark_model_imputed`

`mark_aggregated`, `mark_disaggregated` et `mark_model_imputed` sont des
raccourcis fins autour de `mark_imputed(column, index, provenance)`, qui
écrit directement via `.loc[index, column] = provenance`. `index` accepte
donc tout ce que `.loc` accepte : un timestamp seul, un `DatetimeIndex`,
ou une slice.

### 5.1 - Trois formes d'`index` sur `pib_trimestriel`

In [ ]:
tracker_marks = ImputationProvenanceTracker()
tracker_marks.initialize(df_timeseries)

pib_observed = df_timeseries['pib_trimestriel'].dropna().index
print(f"{len(pib_observed)} valeurs trimestrielles observées, à agréger depuis les mois du trimestre")

# Timestamp unique
tracker_marks.mark_aggregated('pib_trimestriel', pib_observed[0])

# DatetimeIndex (plusieurs dates à la fois)
tracker_marks.mark_aggregated('pib_trimestriel', pib_observed[1:5])

# Slice sur les labels de l'index
tracker_marks.mark_aggregated('pib_trimestriel', slice(pib_observed[5], pib_observed[8]))

display(tracker_marks.get_provenance_matrix().loc[pib_observed[:9], 'pib_trimestriel'])

### 5.2 - `mark_model_imputed` : `trained_on_imputed` bascule entre
`MODEL_ON_TRUE` et `MODEL_ON_IMPUTED`

In [ ]:
inflation_missing = df_timeseries[df_timeseries['inflation_ipc'].isna()].index
print("Dates manquantes pour inflation_ipc :", list(inflation_missing))

# Modèle entraîné uniquement sur données vraies -> MODEL_ON_TRUE (défaut)
tracker_marks.mark_model_imputed('inflation_ipc', inflation_missing, trained_on_imputed=False)
print(tracker_marks.get_provenance('inflation_ipc', inflation_missing))

# Modèle entraîné sur un mélange de vraies et imputées -> MODEL_ON_IMPUTED
tracker_marks.mark_model_imputed('taux_chomage', inflation_missing, trained_on_imputed=True)
print(tracker_marks.get_provenance('taux_chomage', inflation_missing))

### 5.3 - `mark_disaggregated`

Marque les sous-périodes d'une observation basse fréquence étalée sur sa
période, rééchelonnées pour sommer exactement à la valeur observée
(garantie absente des `MODEL_ON_*`).

In [ ]:
# Seules les dates annuelles postérieures au début de la grille mensuelle ont des
# sous-périodes mensuelles présentes dans l'index (les dates antérieures à 2018,
# cf. §2.1, sont des points isolés sans mois autour d'elles)
bc_annual_dates = df_timeseries['balance_commerciale_annuelle'].dropna().index
bc_date = bc_annual_dates[bc_annual_dates >= '2018-01-01'][0]
disagg_months = pd.date_range(bc_date, periods=12, freq='MS')
tracker_marks.mark_disaggregated('balance_commerciale_annuelle', disagg_months)
print(tracker_marks.get_provenance('balance_commerciale_annuelle', disagg_months[:3]))

### 5.4 - Cas limites

- Colonne inconnue -> `ValueError` (avant toute écriture).
- `provenance` qui n'est pas un `ProvenanceType` (ex. la chaîne brute) ->
  `ValueError` : `mark_imputed` n'accepte pas les valeurs `str`
  équivalentes malgré l'héritage `str` de l'énumération.

In [ ]:
try:
    tracker_marks.mark_imputed('colonne_absente', df_timeseries.index[0], ProvenanceType.MODEL_ON_TRUE)
except ValueError as e:
    print("ValueError (colonne inconnue) :", e)

try:
    tracker_marks.mark_imputed('inflation_ipc', df_timeseries.index[0], 'model_on_true')
except ValueError as e:
    print("ValueError (provenance non-ProvenanceType) :", e)

## 6 - `get_provenance(column, index)`

Renvoie un `ProvenanceType` scalaire pour un index unique, une `Series`
pour un index multiple (même comportement d'indexation que `.loc`).

In [ ]:
# Index unique -> scalaire ProvenanceType
single = tracker_marks.get_provenance('pib_trimestriel', pib_observed[0])
print("Index unique :", repr(single), '-> type', type(single).__name__)

# Index multiple -> Series
multiple = tracker_marks.get_provenance('pib_trimestriel', pib_observed[:5])
print("\nIndex multiple -> type", type(multiple).__name__)
print(multiple)

# Colonne inconnue -> ValueError
try:
    tracker_marks.get_provenance('colonne_absente', df_timeseries.index[0])
except ValueError as e:
    print("\nValueError :", e)

## 7 - `get_mask(provenance_types, column=None)`

Masque booléen (`Series` si `column` fourni, `DataFrame` sinon), `True`
là où la provenance appartient à `provenance_types` (normalisé en liste
si un seul `ProvenanceType` est fourni).

In [ ]:
# Un seul type, une seule colonne -> Series booléenne
mask_pib_aggregated = tracker_marks.get_mask(ProvenanceType.AGGREGATED, column='pib_trimestriel')
print("Nb valeurs AGGREGATED sur pib_trimestriel :", mask_pib_aggregated.sum())

# Plusieurs types, toutes colonnes -> DataFrame booléen
mask_model = tracker_marks.get_mask([ProvenanceType.MODEL_ON_TRUE, ProvenanceType.MODEL_ON_IMPUTED])
print("\nType retourné (toutes colonnes) :", type(mask_model).__name__)
print("Nb valeurs MODEL_ON_* par colonne :")
print(mask_model.sum())

# Colonne inconnue -> ValueError
try:
    tracker_marks.get_mask(ProvenanceType.ORIGINAL, column='colonne_absente')
except ValueError as e:
    print("\nValueError :", e)

## 8 - `compute_statistics()`

Calcule, pour `'overall'` puis pour chaque colonne suivie, le compte et
le pourcentage de chaque `ProvenanceType`, plus une entrée `not_imputed`
(cellules encore à `None`). Stocke également le résultat dans
`self.statistics_`.

In [ ]:
stats = tracker_marks.compute_statistics()

print("Clés du dictionnaire :", list(stats.keys()))
print("\n--- overall ---")
for k, v in stats['overall'].items():
    print(f"  {k:20} {v}")

print("\n--- pib_trimestriel ---")
for k, v in stats['pib_trimestriel'].items():
    print(f"  {k:20} {v}")

print("\nstatistics_ est bien mis à jour :", tracker_marks.statistics_ is stats)

Sur le panel, les colonnes de `panel_cols` (ex. `country`) sont bien
absentes des statistiques, comme de la matrice de provenance.

In [ ]:
stats_panel = tracker_panel.compute_statistics()
print("Clés (colonnes suivies + 'overall') :", list(stats_panel.keys()))
print("'country' absente :", 'country' not in stats_panel)

### 8.1 - Cas limite : tracker non initialisé

In [ ]:
try:
    ImputationProvenanceTracker().compute_statistics()
except ValueError as e:
    print("ValueError :", e)

## 9 - `get_provenance_matrix()` et `to_string_matrix()`

`get_provenance_matrix()` renvoie une **copie** (`.copy()`) : la modifier
n'affecte pas l'état interne du tracker. `to_string_matrix()` convertit
chaque cellule en chaîne (`ProvenanceType.value`, ou `'not_imputed'` pour
les cellules encore `None`/`NaN`).

In [ ]:
# Copie : une modification externe ne touche pas le tracker
ref_date, ref_col = pd.Timestamp('2020-01-01'), 'taux_chomage'
print("Valeur avant modification externe :", tracker_marks.get_provenance(ref_col, ref_date))

matrix_copy = tracker_marks.get_provenance_matrix()
matrix_copy.loc[ref_date, ref_col] = 'VALEUR_BIDON'
print("Valeur interne après modification de la copie :", tracker_marks.get_provenance(ref_col, ref_date))

# Représentation en chaînes
string_matrix = tracker_marks.to_string_matrix()
print("\nValeurs uniques (string_matrix) :", sorted(string_matrix.stack().unique().tolist()))
display(string_matrix.loc[pib_observed[:3]])

## 10 - `merge(other, how='update')`

Fusionne deux trackers de même forme :
- `'update'` (défaut) : les valeurs non-`None` de `other` écrasent
  celles de `self`.
- `'preserve'` : seules les valeurs `None` de `self` sont remplies avec
  celles de `other` ; les valeurs déjà présentes dans `self` sont
  conservées telles quelles.

Utile pour combiner la provenance de plusieurs étapes d'imputation (ex.
agrégation puis modèle) sans perdre l'information de la première passe.

In [ ]:
dates4 = df_timeseries.index[:4]
data4 = df_timeseries.loc[dates4, ['pib_trimestriel']]

# 'update' : other écrase self
tracker_a = ImputationProvenanceTracker()
tracker_a.initialize(data4)
tracker_a.mark_model_imputed('pib_trimestriel', dates4[1], trained_on_imputed=False)

tracker_b = ImputationProvenanceTracker()
tracker_b.initialize(data4)
tracker_b.mark_disaggregated('pib_trimestriel', dates4[1])  # même cellule, provenance différente

print("Avant merge (tracker_a) :", list(tracker_a.provenance_matrix_['pib_trimestriel']))
tracker_a.merge(tracker_b, how='update')
print("Après merge 'update'    :", list(tracker_a.provenance_matrix_['pib_trimestriel']), "-> other gagne")

# 'preserve' : self gagne, other ne comble que les None
tracker_c = ImputationProvenanceTracker()
tracker_c.initialize(data4)
tracker_c.mark_model_imputed('pib_trimestriel', dates4[1], trained_on_imputed=False)

tracker_d = ImputationProvenanceTracker()
tracker_d.initialize(data4)
tracker_d.mark_disaggregated('pib_trimestriel', dates4[1])
tracker_d.mark_aggregated('pib_trimestriel', dates4[2])  # None dans tracker_c -> sera comblé

print("\nAvant merge (tracker_c) :", list(tracker_c.provenance_matrix_['pib_trimestriel']))
tracker_c.merge(tracker_d, how='preserve')
print("Après merge 'preserve'  :", list(tracker_c.provenance_matrix_['pib_trimestriel']), "-> dates4[1] conservé, dates4[2] comblé")

### 10.1 - Cas limites

- `self` ou `other` non initialisé -> `ValueError`.
- Formes incompatibles -> `ValueError`.
- `how` invalide -> `ValueError`.
- Après un `merge`, `statistics_` est invalidé (remis à `None`) pour
  forcer un recalcul.

In [ ]:
tracker_uninit = ImputationProvenanceTracker()
try:
    tracker_a.merge(tracker_uninit)
except ValueError as e:
    print("ValueError (other non initialisé) :", e)

tracker_small = ImputationProvenanceTracker()
tracker_small.initialize(data4.iloc[:2])
try:
    tracker_a.merge(tracker_small)
except ValueError as e:
    print("ValueError (formes incompatibles) :", e)

try:
    tracker_a.merge(tracker_b, how='invalide')
except ValueError as e:
    print("ValueError (how invalide) :", e)

tracker_a.compute_statistics()
print("\nstatistics_ avant merge :", tracker_a.statistics_ is not None)
tracker_a.merge(tracker_b, how='update')
print("statistics_ après merge  :", tracker_a.statistics_)

## 11 - `__repr__`

Affiche `'not initialized'` tant qu'`initialize()` n'a pas été appelé,
puis `rows=... cols=...` d'après la forme de la matrice de provenance.

In [ ]:
print(repr(ImputationProvenanceTracker()))
print(repr(tracker_ts))
print(repr(tracker_panel))